# Lab 2 — Time Complexity Analysis
### Advanced Algorithms and Complexity — Master in AI and Master in RSD · Chapter 2

| | |
|---|---|
| **Lectures & Tutorials** | Dr. Benyamina Ahmed |
| **Labs** | Mahkouka Mohamed Fouad |
| **Duration** | 1 session |

---
## 🎯 Objective

Put chapter 2 into practice in three parts, as on the lab sheet:

| Part | Sheet | Goal |
|---|---|---|
| 1 | Lab 1 — Experimental measurement | implement 4 algorithms and measure their execution times |
| 2 | Lab 2 — Visualising complexities | plot the theoretical curves $1$, $\log_2 n$, $n$, $n\log_2 n$, $n^2$ |
| 3 | Lab 3 — Mini-project | compare 4 sorting algorithms, from implementation to conclusion |

In each part: ✍️ you **program** the functions, ✅ a cell **checks** they are correct, 📈 the **measurement and plotting** cells are provided, then you complete a few ✍️ lines.

## 🖥️ Environment

Language: Python 3 · Tools: `time`, `matplotlib`. Write down your **machine** (CPU, RAM) and **OS**:

> ✍️ **Machine:** … · **OS:** …

> 💡 Close unnecessary applications during measurements. Printing and data generation are **never** included in the measured time.

---
# Tools

**Run this cell first.** You do not need to change it.

| Function | Purpose |
|---|---|
| `check(name, got, expected)` | prints ✅ or ❌ |
| `timer(f, *args)` | **average** time of `f(*args)` over **5 runs** |
| `slope(sizes, values)` | log-log slope: ≈ 0 → $\log n$, ≈ 1 → $n$, a bit above 1 → $n\log n$, ≈ 2 → $n^2$ |
| `show_table`, `plot`, `save_csv` | table, log-log graph (PNG), CSV file |

In [ ]:
def check(name, got, expected):
    print(("✅" if got == expected else "❌"), name, "| got :", got, "| expected :", expected)


import csv
import math
import random
import time
from statistics import linear_regression
import matplotlib.pyplot as plt


def timer(f, *args, repetitions=5):
    # Average time (seconds) of f(*args) over 5 runs.
    total = 0.0
    for _ in range(repetitions):
        start = time.perf_counter()
        f(*args)
        total += time.perf_counter() - start
    return total / repetitions


def slope(sizes, values):
    # Slope of the log-log line, computed on the measured points (None ignored).
    pts = [(n, v) for n, v in zip(sizes, values) if v]
    lx = [math.log10(n) for n, _ in pts]
    ly = [math.log10(v) for _, v in pts]
    return round(linear_regression(lx, ly).slope, 2)


def show_table(sizes, results):
    # results = {name: [value or None]}
    names = list(results)
    print(f"{'n':>10} | " + " | ".join(f"{c:>20}" for c in names))
    print("-" * (13 + 23 * len(names)))
    for i, n in enumerate(sizes):
        row = [results[c][i] if i < len(results[c]) else None for c in names]
        cells = [f"{v:>20.3e}" if v is not None else f"{'—':>20}" for v in row]
        print(f"{n:>10,} | " + " | ".join(cells))


def plot(title, sizes, results, ylabel="time (s)", file=None):
    # Log-log curves; saved as PNG if a file name is given.
    plt.figure(figsize=(8, 5))
    for name, values in results.items():
        pts = [(n, v) for n, v in zip(sizes, values) if v]
        if pts:
            plt.loglog([p[0] for p in pts], [p[1] for p in pts], "o-", label=name)
    plt.xlabel("input size n (log)")
    plt.ylabel(ylabel)
    plt.title(title)
    plt.grid(True, which="both", alpha=0.3)
    plt.legend()
    if file:
        plt.savefig(file, dpi=120, bbox_inches="tight")
    plt.show()


def save_csv(file, sizes, results):
    with open(file, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["n"] + list(results))
        for i, n in enumerate(sizes):
            w.writerow([n] + [results[c][i] if i < len(results[c]) else None for c in results])
    print("Table saved:", file)


print("Tools ready ✅")

---
# Part 1 — Experimental measurement of complexity

**Goal:** implement four simple algorithms and measure their execution times for different sizes $n$.

| # | Algorithm | Theoretical complexity |
|---|---|---|
| 1 | Linear search (unsorted array) | $O(n)$ |
| 2 | Binary search (**sorted** array) | $O(\log n)$ |
| 3 | Insertion sort | $O(n^2)$ |
| 4 | Merge sort | $O(n \log n)$ |

> 💡 Tip from the sheet: use the same **interface** for every function. Searches take `(T, x)` and return the **index** of `x` (or `-1`). Sorts take `T` and return a **new sorted list** (without changing `T`, without using `sorted`).

## 1.1 Linear search
Go through `T` from left to right; return the index of the first cell equal to `x`, or `-1`.

In [ ]:
def linear_search(T, x):
    for i in range(len(T)):
        if T[i] == x:
            return i
    return -1

check("found", linear_search([7, 3, 9, 1], 9), 2)
check("absent", linear_search([7, 3, 9, 1], 5), -1)
check("empty list", linear_search([], 5), -1)

## 1.2 Binary search
In a **sorted** array: look at the middle cell; if `x` is larger, search the right half, otherwise the left half. Stop when the search area is empty.

```
low ← 0 ; high ← n - 1
while low ≤ high
    mid ← (low + high) // 2
    if T[mid] = x then return mid
    else if T[mid] < x then low ← mid + 1
    else high ← mid - 1
return -1
```

In [ ]:
def binary_search(T, x):
    low, high = 0, len(T) - 1
    while low <= high:
        mid = (low + high) // 2
        if T[mid] == x:
            return mid
        elif T[mid] < x:
            low = mid + 1
        else:
            high = mid - 1
    return -1

T = [1, 3, 5, 7, 9, 11]
check("x = 1", binary_search(T, 1), 0)
check("x = 11", binary_search(T, 11), 5)
check("x = 7", binary_search(T, 7), 3)
check("absent", binary_search(T, 4), -1)

## 1.3 Insertion sort
For each element from the second one: take it out (`key`), shift larger elements one cell to the right, then insert it at its place.

```
for i = 1 to n - 1
    key ← T[i] ; j ← i - 1
    while j ≥ 0 and T[j] > key
        T[j + 1] ← T[j] ; j ← j - 1
    T[j + 1] ← key
```

In [ ]:
def insertion_sort(T):
    T = T[:]          # copy: the original list is not changed
    for i in range(1, len(T)):
        key = T[i]
        j = i - 1
        while j >= 0 and T[j] > key:
            T[j + 1] = T[j]
            j -= 1
        T[j + 1] = key
    return T

random.seed(0)
data = [random.randint(0, 100) for _ in range(50)]
check("50 integers sorted", insertion_sort(data) == sorted(data), True)
check("empty list", insertion_sort([]), [])
check("already sorted", insertion_sort([1, 2, 3]), [1, 2, 3])

## 1.4 Merge sort
1. `merge(L, R)` **merges** two sorted lists into one sorted list: compare the two first elements, take the smaller one, repeat; at the end, add what is left.
2. `merge_sort(T)`: if `T` has 0 or 1 element it is sorted; otherwise split it into two halves, sort each half (recursion), then merge them.

In [ ]:
def merge(L, R):
    result = []
    i = j = 0
    while i < len(L) and j < len(R):
        if L[i] <= R[j]:
            result.append(L[i]); i += 1
        else:
            result.append(R[j]); j += 1
    result.extend(L[i:])
    result.extend(R[j:])
    return result

def merge_sort(T):
    if len(T) <= 1:
        return T[:]
    m = len(T) // 2
    return merge(merge_sort(T[:m]), merge_sort(T[m:]))

check("merge", merge([1, 4, 9], [2, 3, 10, 11]), [1, 2, 3, 4, 9, 10, 11])
check("50 integers sorted", merge_sort(data) == sorted(data), True)
check("empty list", merge_sort([]), [])

## 1.5 Measurements

Method from the sheet: for each $n \in \{10, 100, \dots, 10^6\}$ and each algorithm, generate the data, run **5 times**, keep the **average time**.

- Searches: **worst case**, `x` absent (`x = -1`). Binary search works on the **sorted** version of `T` (sorting happens **before** timing).
- ⚠️ In Python, insertion sort is limited to $n \le 10{,}000$ ($10^5$ would take several minutes per measurement) and merge sort to $n \le 100{,}000$. Cells not measured hold `None` and show as "—".

✍️ Complete the loop: linear search is done, add the three other algorithms.

In [ ]:
sizes = [10, 100, 1_000, 10_000, 100_000, 1_000_000]
results1 = {"Linear search": [], "Binary search": [],
            "Insertion sort": [], "Merge sort": []}

random.seed(1)
for n in sizes:
    T = [random.randint(0, 10 * n) for _ in range(n)]
    T_sorted = sorted(T)
    results1["Linear search"].append(timer(linear_search, T, -1))
    results1["Binary search"].append(timer(binary_search, T_sorted, -1))
    results1["Insertion sort"].append(timer(insertion_sort, T) if n <= 10_000 else None)
    results1["Merge sort"].append(timer(merge_sort, T) if n <= 100_000 else None)

show_table(sizes, results1)

In [ ]:
for name, values in results1.items():
    if len(values) == len(sizes):
        print(f"{name:<22} slope ≈ {slope(sizes[2:], values[2:])}")   # from n = 1000
plot("Part 1 — average times (log-log)", sizes, results1, file="tp2_part1.png")
save_csv("results_tp2_part1.csv", sizes, results1)

### Comparison with theoretical complexities

| Algorithm | Theoretical complexity | Measured slope | Observed behaviour | Matches? |
|---|---|---|---|---|
| Linear search | $O(n)$ | ≈ 1 | linear growth | ✅ |
| Binary search | $O(\log n)$ | ≈ 0.1 | very slow growth, almost flat | ✅ |
| Insertion sort | $O(n^2)$ | ≈ 2 | quadratic growth | ✅ |
| Merge sort | $O(n \log n)$ | ≈ 1 to 1.2 | between linear and quadratic | ✅ |

Exact slopes depend on the machine.

> ✍️ **Conclusion (2 lines):** the measurements confirm $O(\log n) < O(n) < O(n \log n) < O(n^2)$. Constants and the machine change the times, but not the order of growth; from $n \ge 10^4$ the gaps become very large.

---
# Part 2 — Visualising complexities

**Objective:** plot the **theoretical** curves of common complexities as a function of $n$ (log scale) and compare them.

| Function | Name |
|---|---|
| $O(1) = 1$ | constant |
| $O(\log n) = \log_2 n$ | logarithmic |
| $O(n) = n$ | linear |
| $O(n \log n) = n \log_2 n$ | quasi-linear |
| $O(n^2) = n^2$ | quadratic |

✍️ For $n = 10, 10^2, \dots, 10^6$, compute the value of each function and put it in the matching list of `values`. Use `math.log2`.

In [ ]:
ns = [10**k for k in range(1, 7)]
values = {"O(1)": [], "O(log n)": [], "O(n)": [], "O(n log n)": [], "O(n²)": []}

for n in ns:
    values["O(1)"].append(1)
    values["O(log n)"].append(math.log2(n))
    values["O(n)"].append(n)
    values["O(n log n)"].append(n * math.log2(n))
    values["O(n²)"].append(n ** 2)

check("O(log n) for n = 10", round(values["O(log n)"][0], 2) if values["O(log n)"] else None, 3.32)
check("O(n log n) for n = 10⁶", round(values["O(n log n)"][-1], 2) if values["O(n log n)"] else None, 19931568.57)

In [ ]:
# Table of theoretical values, graph (PNG) and CSV file
print(f"{'n':>9} | " + " | ".join(f"{k:>14}" for k in values))
for i, n in enumerate(ns):
    print(f"{n:>9,} | " + " | ".join(f"{values[k][i]:>14,.2f}" if values[k] else f"{'—':>14}" for k in values))

plot("Comparison of theoretical complexities", ns, values,
       ylabel="value (relative)", file="tp2_complexities.png")
save_csv("tp2_values.csv", ns, values)

### Question 3 in code
"From which value of $n$ does $O(n^2)$ become much larger than $O(n)$?"
We define "much larger" as **at least 100 times** larger. ✍️ Write a `while` loop that finds the first $n$ such that $n^2 \ge 100 \cdot n$.

In [ ]:
n = 1
while n ** 2 < 100 * n:
    n += 1

check("first n", n, 100)

> ✍️ **Q1. Which function grows the slowest?** $O(1)$ does not grow; among the others, $\log_2 n$ grows the slowest (almost flat).

> ✍️ **Q2. Which function grows the fastest?** $n^2$ (slope ≈ 2 on a log-log scale).

> ✍️ **Q4. How does the log-log scale help compare complexities?** values from $1$ to $10^{12}$ fit on one graph; each $n^k$ becomes a straight line with slope $k$, and a constant factor becomes a simple vertical shift.

---
# Part 3 — Mini-project: sorting an array of $n$ random integers

**Objective:** compare several sorting algorithms by execution time and validate their theoretical complexities, following every step: implementation → measurement → comparison → conclusion.

| Algorithm | Idea | Time | Space |
|---|---|---|---|
| Insertion sort | inserts each element at its place in the sorted part | $O(n^2)$ | $O(1)$ |
| Merge sort | splits then merges the sub-arrays | $O(n\log n)$ | $O(n)$ |
| Quick sort | partitions around a pivot | $O(n\log n)$ on average, $O(n^2)$ worst case | $O(\log n)$ on average |
| Heap sort | uses a binary heap | $O(n\log n)$ | $O(1)$ |

Insertion sort and merge sort are already written (Part 1).

## 3.1 Quick sort
✍️ Use the **middle** element as the pivot. Build three lists: elements **smaller** than the pivot, **equal**, **larger**. Return `sort(smaller) + equal + sort(larger)`.

In [ ]:
def quick_sort(T):
    if len(T) <= 1:
        return T[:]
    pivot = T[len(T) // 2]
    smaller = [x for x in T if x < pivot]
    equal = [x for x in T if x == pivot]
    larger = [x for x in T if x > pivot]
    return quick_sort(smaller) + equal + quick_sort(larger)

check("50 integers sorted", quick_sort(data) == sorted(data), True)
check("with duplicates", quick_sort([3, 1, 3, 2, 1]), [1, 1, 2, 3, 3])

## 3.2 Heap sort *(provided: read it and run it)*
A **max-heap** is a binary tree stored in an array: the children of cell `i` are `2i + 1` and `2i + 2`, and every parent is larger than its children. So the largest element is `T[0]`. We swap it with the last one, shrink the heap by one cell, and let the new root "sink down" (`sift_down`).

In [ ]:
def sift_down(T, i, end):
    # moves T[i] down while a child (before end) is larger
    while 2 * i + 1 < end:
        child = 2 * i + 1
        if child + 1 < end and T[child + 1] > T[child]:
            child += 1
        if T[i] >= T[child]:
            return
        T[i], T[child] = T[child], T[i]
        i = child

def heap_sort(T):
    T = T[:]
    n = len(T)
    for i in range(n // 2 - 1, -1, -1):     # build the max-heap
        sift_down(T, i, n)
    for end in range(n - 1, 0, -1):         # extract the maximum, one by one
        T[0], T[end] = T[end], T[0]
        sift_down(T, 0, end)
    return T

check("50 integers sorted", heap_sort(data) == sorted(data), True)

## 3.3 Measurement protocol
For each $n \in \{10^2, 10^3, 10^4, 10^5\}$: generate **5 random arrays**, run each algorithm on each array, keep the **average time** (`time.perf_counter()`). Insertion sort stops at $n = 10^4$.

✍️ Complete the **"Best"** column: for each $n$, the name of the fastest algorithm.

In [ ]:
sorts = {"Insertion": insertion_sort, "Merge": merge_sort, "Quick": quick_sort, "Heap": heap_sort}
sizes3 = [100, 1_000, 10_000, 100_000]
results3 = {name: [] for name in sorts}

random.seed(3)
for n in sizes3:
    arrays = [[random.randint(0, 10 * n) for _ in range(n)] for _ in range(5)]
    for name, f in sorts.items():
        if name == "Insertion" and n > 10_000:
            results3[name].append(None)
        else:
            results3[name].append(sum(timer(f, A, repetitions=1) for A in arrays) / 5)

show_table(sizes3, results3)

In [ ]:
best = []
for i, n in enumerate(sizes3):
    measured = {name: results3[name][i] for name in results3 if results3[name][i] is not None}
    best.append(min(measured, key=measured.get))

print("Best :", best)

In [ ]:
for name, values in results3.items():
    print(f"{name:<10} slope ≈ {slope(sizes3, values)}")
plot("Part 3 — sorting algorithms (log-log)", sizes3, results3, file="tp2_part3.png")
save_csv("results_tp2_part3.csv", sizes3, results3)

## 3.4 Analysis and comparison

| Algorithm | Theoretical complexity | Measured slope | Observed behaviour | Matches? |
|---|---|---|---|---|
| Insertion | $O(n^2)$ | ≈ 2 | very steep curve, impractical from $n \approx 10^5$ | ✅ |
| Merge | $O(n\log n)$ | ≈ 1.1 | moderate, steady growth | ✅ |
| Quick (average) | $O(n\log n)$ | ≈ 1.1 | often the fastest on random data | ✅ |
| Heap | $O(n\log n)$ | ≈ 1.1 | close to merge sort, a bit slower in Python | ✅ |

The order between merge, quick and heap sort can change from one machine to another: their curves are parallel, only the **constants** differ.

> ✍️ **Reasoned conclusion (3 to 5 lines):** the measurements confirm $O(n^2) \gg O(n\log n)$: insertion sort becomes impractical for large sizes, while the three $n \log n$ sorts stay fast. Between them, the gap only comes from constants and implementation. In the worst case (bad pivot) quick sort can become $O(n^2)$, but on random data it stays $O(n\log n)$. The choice depends on the data size and available memory (merge sort uses $O(n)$ extra memory).

---
# 📤 What to submit

Deliverables from the sheet:

- this completed notebook (it replaces the `.py` file): **Restart & Run All** with no errors, every check ✅;
- the tables `results_tp2_part1.csv`, `tp2_values.csv`, `results_tp2_part3.csv`;
- the graphs `tp2_part1.png`, `tp2_complexities.png`, `tp2_part3.png`;
- the ✍️ analyses and conclusions (theory vs experiment).

```bash
git add tp2_complexity/
git commit -m "Lab 2: time complexity analysis"
git push
```

## 🎯 Key takeaways

- Experimental measurements **confirm (or refute)** theoretical predictions.
- Constants and the machine change the measured times, **not the order of growth**.
- On a log-log scale, parallel lines mean the same complexity; the **slope** gives the degree.
- $O(1) < O(\log n) < O(n) < O(n\log n) < O(n^2)$.

### ➡️ Next session: **Lab 3 — Binary search trees**